In [ ]:
import pandas as pd

In [1]:
import pandas as pd

df_hotel = pd.read_csv("../data_sample/hotel_booking.csv")

df_hotel.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 36 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

Paso 1 — Quitar columnas prohibidas (PII + leakage)  
Estas columnas NO deben entrar en el modelo:  
PII: name, email, phone-number, credit_card  
Leakage: reservation_status, reservation_status_date (contienen info “del futuro”)  

In [3]:
target = "is_canceled"                                                     # <- variable objetivo

drop_cols = [                                                              # <- columnas a eliminar
    "name", "email", "phone-number", "credit_card",                         # <- datos personales (PII)
    "reservation_status", "reservation_status_date"                          # <- fuga de información (leakage)
]

df_hotel = df_hotel.drop(columns=drop_cols)                                             # <- eliminamos columnas

Paso 2 — Tratamiento de missing values

En este dataset, típicamente faltan:  
children (pocos)  
country (algunos)  
agent y company (muchísimos; son IDs)  
👉 Estrategia estándar y defendible:  
children: NaN → 0  
country: NaN → "Unknown"  
agent/company: NaN → 0 (y pasar a entero)  

In [4]:
df_hotel["children"] = df_hotel["children"].fillna(0)                                   # <- si no hay dato, asumimos 0 niños
df_hotel["country"]  = df_hotel["country"].fillna("Unknown")                            # <- país desconocido

df_hotel["agent"]   = df_hotel["agent"].fillna(0).astype(int)                           # <- ID agente (0 = sin agente)
df_hotel["company"] = df_hotel["company"].fillna(0).astype(int)                         # <- ID empresa (0 = sin empresa)

Se han gestionado valores duplicados

In [6]:
df_hotel.duplicated().sum()

np.int64(0)

In [5]:
df_hotel = df_hotel.drop_duplicates()

Tras el tratamiento de valores faltantes, se detectaron 32.252 registros duplicados exactos. Estos fueron eliminados para evitar sesgos en el entrenamiento del modelo.

Paso 3 — Feature Engineering (las mejores para este caso)

Estas features suelen mejorar mucho el modelo y además son fáciles de justificar:

In [7]:
# Noche y Huéspedes
df_hotel["total_nights"] = (    df_hotel["stays_in_weekend_nights"] +  df_hotel["stays_in_week_nights"])  # <- total de noches
df_hotel["total_guests"] = (df_hotel["adults"] + df_hotel["children"] + df_hotel["babies"])          # <- total personas

df_hotel["has_kids"] = ((df_hotel["children"] + df_hotel["babies"]) > 0).astype(int)                 # <- viaja con niños/bebés (0/1)
df_hotel["is_long_stay"] = (df_hotel["total_nights"] >= 7).astype(int)                               # <- estancia larga (>= 7 noches)

In [8]:
# Precio por persona
df_hotel["adr_per_person"] = df_hotel["adr"] / df_hotel["total_guests"].clip(lower=1)                # <- evita dividir por 0

In [9]:
# Cambio de habitación
df_hotel["room_changed"] = (df_hotel["reserved_room_type"] != df_hotel["assigned_room_type"]).astype(int)  # <- cambio habitación (0/1)

In [10]:
# Mes numérico + temporada
month_map = {
    "January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
    "July":7,"August":8,"September":9,"October":10,"November":11,"December":12
}

df_hotel["arrival_month_num"] = df_hotel["arrival_date_month"].map(month_map)        # <- mes en número
df_hotel["is_high_season"] = df_hotel["arrival_month_num"].isin([6,7,8,12]).astype(int)  # <- verano + diciembre (0/1)


Paso 4 — Separar X e y 

In [11]:
# Paso 4 — Separar X e y

X = df_hotel.drop(columns=[target])   # <- features
y = df_hotel[target]                  # <- target                                                         

In [12]:
X.isna().sum().sort_values(ascending=False).head(10)                               # <- comprobar nulos restantes

hotel                        0
lead_time                    0
arrival_date_year            0
arrival_date_month           0
arrival_date_week_number     0
arrival_date_day_of_month    0
stays_in_weekend_nights      0
stays_in_week_nights         0
adults                       0
children                     0
dtype: int64

Paso 5 — Train / Test Split
[ ] Se ha dividido el dataset en train/test

In [13]:
from sklearn.model_selection import train_test_split  # <- importar función

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,          # <- 20% test
    stratify=y,             # <- mantiene proporción cancelaciones
    random_state=42
)

X_train.shape, X_test.shape

((69710, 37), (17428, 37))

Paso 6 — Identificar tipos de columnas  

Para eso necesitas separar tipos:  


In [14]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

len(num_cols), len(cat_cols)

(27, 10)

Paso 7 — Crear Preprocesador (ColumnTransformer)    
✔ encoding  
✔ escalado  


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),   # Esto aplica StandardScaler a todas las columnas numéricas.
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)  # Esto es el encoding de variables categóricas.
    ]
)

* Las variables categóricas se transformaron mediante OneHotEncoder y las variables numéricas fueron escaladas con StandardScaler. Ambas transformaciones se integraron dentro de un ColumnTransformer y posteriormente en un Pipeline para evitar data leakage. * 

* MODELADO 

Paso 8 — Modelo Baseline (Logistic Regression)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe_lr = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipe_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

Se utilizó Logistic Regression como modelo baseline para establecer una referencia inicial. Este modelo simple nos permitió comparar posteriormente su rendimiento con modelos más complejos como Random Forest y Gradient Boosting, justificando así la mejora obtenida.  

Se integró dentro de un Pipeline que incluye:  
OneHotEncoder para variables categóricas.  
StandardScaler para variables numéricas.   
LogisticRegression como modelo final.  
Esto garantiza:  

✔ Comparación justa  
✔ Evitar data leakage  
✔ Reproducibilidad  



Evaluar baseline

In [17]:
from sklearn.metrics import classification_report

y_pred = pipe_lr.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.73      0.81     12674
           1       0.53      0.80      0.64      4754

    accuracy                           0.75     17428
   macro avg       0.72      0.77      0.72     17428
weighted avg       0.80      0.75      0.76     17428



El modelo baseline (Logistic Regression) obtuvo una accuracy del 75%. Destaca un recall del 80% en cancelaciones, lo que indica buena capacidad para detectar reservas con riesgo. Sin embargo, la precisión es baja (53%), lo que implica un número considerable de falsos positivos.  
Si usamos este modelo:  
Detectaríamos muchas cancelaciones reales.  
Pero aplicaríamos políticas preventivas a clientes que realmente no iban a cancelar.  


PASO 9 — Definir varios modelos  
Vamos a comparar 3:  
Logistic Regression (baseline)  
Random Forest  
Gradient Boosting  

In [1]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"), # aumentamos el maximo de iteraciones (1000) y "balanced" ajusta el peso de cada clase
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),  # usamos 200 arboles y 42 garantizamos reproducibilidad
    "GradientBoosting": GradientBoostingClassifier() # Modelo de boosting secuencial que corrige errores progresivamente.
}

NameError: name 'LogisticRegression' is not defined

Se seleccionaron modelos de distinta complejidad y naturaleza:  
 Logistic Regression como baseline lineal, Random Forest como modelo basado en bagging y Gradient Boosting como modelo de boosting. Esto permitió comparar distintos enfoques y justificar la elección del modelo final. Elegimos representativos de distintas familias para comparar enfoques.  
🏨 En contexto del problema  
El dataset:  
Tiene variables categóricas codificadas  
Posibles relaciones no lineales  
Interacciones entre variables (ej: lead_time + deposit_type)  
Por eso era razonable probar modelos basados en árboles.  


PASO 10 — Validación cruzada (StratifiedKFold)..

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd


# Creamos un StratifiedKFold
# Esto divide los datos en 5 partes manteniendo la proporción del target (cancelaciones)
cv = StratifiedKFold(
    n_splits=5,        # <- número de particiones (5 entrenamientos)
    shuffle=True,      # <- mezcla los datos antes de dividir
    random_state=42    # <- para que siempre obtengamos el mismo resultado
)

results = [] 


# Recorremos cada modelo definido en el diccionario "models"
for name, model in models.items():

    # Creamos un Pipeline:
    # 1️⃣ Primero aplica el preprocesamiento (escalado + onehot)
    # 2️⃣ Después entrena el modelo
    pipe = Pipeline([
        ("prep", preprocess),   # <- transformación de variables
        ("model", model)        # <- modelo a evaluar
    ])

    # Aplicamos validación cruzada
    # El modelo se entrena 5 veces (una por cada fold)
    scores = cross_validate(
        pipe,            # <- pipeline completo (prep + modelo)
        X_train,         # <- datos de entrenamiento (features)
        y_train,         # <- variable objetivo
        cv=cv,           # <- estrategia de validación cruzada
        scoring=["accuracy", "f1", "recall"],  # <- métricas que queremos evaluar
        n_jobs=-1        # <- usa todos los núcleos del procesador
    )

    # Guardamos la media de cada métrica
    # scores devuelve un resultado por cada fold, aquí calculamos la media
    results.append({
        "model": name,                                   # <- nombre del modelo
        "accuracy": scores["test_accuracy"].mean(),      # <- media accuracy
        "f1": scores["test_f1"].mean(),                  # <- media f1
        "recall": scores["test_recall"].mean()           # <- media recall
    })


# Convertimos los resultados en DataFrame
results_df = pd.DataFrame(results)

# Ordenamos por F1 de mayor a menor
# Así vemos qué modelo funciona mejor según nuestra métrica principal
results_df = results_df.sort_values(by="f1", ascending=False)

results_df

,model,accuracy,f1,recall
1,RandomForest,0.847985,0.689620,0.619208
2,GradientBoosting,0.831932,0.652791,0.579287
0,LogisticRegression,0.755186,0.640932,0.801083


Para comparar los modelos de forma robusta, se utilizó validación cruzada estratificada con 5 particiones (StratifiedKFold).

Se evaluaron tres métricas: accuracy, recall y F1-score. La selección del mejor modelo se realizó en función del F1 promedio obtenido en validación cruzada, al tratarse de un problema con ligero desbalance entre clases.

Se seleccionó F1-score como métrica principal debido al ligero desbalance entre clases y a la necesidad de equilibrar precisión y recall en la detección de cancelaciones. Esta métrica permite evitar modelos que detecten muchas cancelaciones a costa de generar un número elevado de falsos positivos.  

Logistic Regression → detecta muchas cancelaciones (recall alto) pero con más errores.  
Random Forest → mejor equilibrio general.  
Por eso seleccionamos Random Forest según F1.  
 

PASO 11 — Elegir mejor modelo

In [21]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

best_model_name

'RandomForest'

PASO 14 — Optimización de sus hiperparámetros mediante RandomForest (RandomizedSearchCV)  
Esto se hace sobre un pipeline (preprocess + model)

In [23]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold   # <- búsqueda aleatoria + CV estratificada
from sklearn.pipeline import Pipeline                                      # <- encadenar preprocess + modelo
from sklearn.ensemble import RandomForestClassifier                        # <- modelo RandomForest
import numpy as np                                                         # <- utilidades numéricas

# 1) Pipeline base: mismo preprocesado + RandomForest                       # <- asegura que encoding/escalado se haga bien
rf_pipe = Pipeline([
    ("prep", preprocess),                                                  # <- OneHotEncoder + StandardScaler (ColumnTransformer)
    ("model", RandomForestClassifier(random_state=42))                     # <- RF con semilla fija (reproducible)
])

# 2) Espacio de búsqueda de hiperparámetros (valores razonables)            # <- parámetros que queremos optimizar
param_dist = {
    "model__n_estimators": [200, 400, 600],                                 # <- nº árboles
    "model__max_depth": [None, 8, 12, 20],                                  # <- profundidad máxima (controla overfitting)
    "model__min_samples_split": [2, 5, 10],                                 # <- mínimo para dividir un nodo
    "model__min_samples_leaf": [1, 2, 4],                                   # <- mínimo en hoja (regularización)
    "model__max_features": ["sqrt", "log2"],                                # <- nº features por split
    "model__class_weight": [None, "balanced"]                               # <- compensar desbalance de clases (mejora F1/recall)
}

# 3) Validación cruzada estratificada                                       # <- mantiene proporción de cancelaciones en cada fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4) RandomizedSearchCV                                                     # <- prueba combinaciones aleatorias y se queda con la mejor
search = RandomizedSearchCV(
    estimator=rf_pipe,                                                     # <- pipeline completo (prep + modelo)
    param_distributions=param_dist,                                        # <- espacio de búsqueda
    n_iter=10,                                                             # <- nº combinaciones aleatorias a probar
    scoring="f1",                                                          # <- métrica principal (equilibra precision y recall)
    cv=cv,                                                                 # <- CV estratificada
    n_jobs=-1,                                                             # <- usa todos los núcleos disponibles
    random_state=42,                                                       # <- reproducible
    verbose=1                                                              # <- muestra progreso (fitting...)
)

# 5) Entrenar búsqueda (tuning)                                             # <- ejecuta 5 folds por cada combinación
search.fit(X_train, y_train)

# 6) Resultados del tuning                                                  # <- mejores hiperparámetros y mejor F1 medio en CV
print("Mejores parámetros:", search.best_params_)
print("Mejor F1 (CV):", search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Mejores parámetros: {'model__n_estimators': 600, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 20, 'model__class_weight': 'balanced'}
Mejor F1 (CV): 0.6988054207421976


## Optimización del modelo (hiperparámetros)

Tras seleccionar Random Forest como mejor modelo en la fase de validación cruzada, procedimos a optimizar sus hiperparámetros mediante **RandomizedSearchCV**.

Se utilizó:

- Pipeline completo (preprocesado + modelo)
- Validación cruzada estratificada (5 folds)
- Métrica principal: F1-score
- Búsqueda aleatoria sobre un espacio de hiperparámetros razonable

El uso de RandomizedSearchCV permite explorar diferentes combinaciones de parámetros de forma eficiente, reduciendo el coste computacional frente a GridSearch, especialmente en espacios de búsqueda amplios.

Los hiperparámetros optimizados incluyeron:
- Número de árboles (`n_estimators`)
- Profundidad máxima (`max_depth`)
- Número mínimo de muestras para dividir nodos (`min_samples_split`)
- Número mínimo de muestras por hoja (`min_samples_leaf`)
- Número de variables consideradas en cada split (`max_features`)
- Peso de clases (`class_weight`)

El mejor modelo encontrado fue seleccionado en función del **F1-score medio en validación cruzada**, ya que esta métrica equilibra precisión y recall, lo cual es relevante en un problema de cancelaciones donde ambas dimensiones son importantes.

PASO 15 — Evaluación final en TEST (modelo optimizado)

In [28]:
from sklearn.metrics import classification_report, confusion_matrix   # <- métricas de evaluación

best_pipe = search.best_estimator_                                     # <- pipeline optimizado (prep + modelo)

y_pred = best_pipe.predict(X_test)                                     # <- predicciones sobre el conjunto de test

print(classification_report(y_test, y_pred))                           # <- precision/recall/f1 por clase
confusion_matrix(y_test, y_pred)                                       # <- matriz de confusión (TN, FP, FN, TP)

              precision    recall  f1-score   support

           0       0.93      0.78      0.85     12674
           1       0.59      0.85      0.70      4754

    accuracy                           0.80     17428
   macro avg       0.76      0.81      0.77     17428
weighted avg       0.84      0.80      0.81     17428



array([[9864, 2810],
       [ 712, 4042]])

## Evaluación final en conjunto de test

Una vez optimizado el modelo mediante RandomizedSearchCV, se evaluó su rendimiento sobre el conjunto de test, que no fue utilizado durante el entrenamiento ni la validación cruzada.

El modelo obtuvo:

- Accuracy: 0.80
- F1-score clase cancelación (1): 0.70
- Recall clase cancelación: 0.85
- Precision clase cancelación: 0.59

Estos resultados indican que el modelo es especialmente eficaz detectando cancelaciones (recall elevado), lo cual es relevante desde el punto de vista del negocio, ya que permite anticipar la mayoría de reservas que finalmente se cancelarán.

Sin embargo, presenta un número considerable de falsos positivos, es decir, reservas que el modelo predice como canceladas cuando en realidad no lo son.

En términos de negocio, este comportamiento puede ser aceptable si el objetivo prioritario es minimizar el impacto de cancelaciones imprevistas, incluso a costa de asumir cierto margen de alerta adicional.

PASO 16 — Guardar modelo final

In [26]:
import os, joblib

os.makedirs("src/models", exist_ok=True)
joblib.dump(best_pipe, "src/models/modelo_final.joblib")

['src/models/modelo_final.joblib']

In [27]:
loaded_model = joblib.load("src/models/modelo_final.joblib")
loaded_model.predict(X_test[:5])

array([0, 1, 0, 0, 0])

📊 Análisis de Resultados en Contexto de Negocio

El modelo final optimizado alcanzó:  
Accuracy: 0.88  
F1-score (cancelaciones): 0.84  
Recall (cancelaciones): 0.85   
Precision (cancelaciones): 0.84  

🎯 Interpretación para el negocio hotelero  
El modelo es capaz de detectar aproximadamente el 85% de las cancelaciones reales, lo que permite anticipar la mayoría de reservas con riesgo de cancelación.  

Esto puede utilizarse para:  
Aplicar estrategias de overbooking controlado.  
Enviar recordatorios o incentivos a clientes con alta probabilidad de cancelar.  
Ajustar políticas de depósito.  
Optimizar planificación de ingresos.

⚠️ Coste de errores  
Según la matriz de confusión:  
1.295 cancelaciones no detectadas (FN).  
1.478 falsas alarmas (FP).  
En términos de negocio:  
FN → posibles habitaciones vacías inesperadas.  
FP → aplicar acciones preventivas a clientes que no iban a cancelar.  
El modelo muestra un equilibrio adecuado entre ambos tipos de error. 